# Seattle : nettoyage des données (2016)

Ce notebook applique les décisions prises dans `01_exploration.ipynb`
(partie 5, bilan des décisions). On repart du fichier brut, on applique
chaque décision dans le même ordre que l'exploration, puis on exporte un
fichier propre pour la modélisation.

Après chaque étape, on affiche le nombre de lignes et de colonnes, pour
suivre ce qu'on enlève.

**Plan du notebook**

1. Chargement et périmètre
2. Valeurs manquantes
3. Erreurs et incohérences
4. Fuite de données
5. Vérifications
6. Export

**Ce qu'on ne fait pas ici** : tout ce qui demande un calcul statistique
(remplir avec la médiane, mise à l'échelle, encodage). Ce sera fait dans le
pipeline de modélisation, après la séparation entre entraînement et test,
pour éviter une fuite de données.

## Imports et chemins

In [1]:
from pathlib import Path

import pandas as pd

# Chemins
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# Options d'affichage
pd.set_option("display.max_columns", None)

In [2]:
def afficher_taille(df, etape):
    """Affiche le nombre de lignes et de colonnes après une étape."""
    print(f"{etape} : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# 1. Chargement et périmètre

## 1.1. Chargement

In [3]:
df_raw = pd.read_csv(
    DATA_RAW / "2016-building-energy-benchmarking-6af870b7ba821669818300.csv"
)
afficher_taille(df_raw, "Fichier brut")

Fichier brut : 3376 lignes, 46 colonnes


## 1.2. Périmètre : les bâtiments non résidentiels

*Exploration, partie 1.3* : on exclut les logements, repérés avec
`BuildingType` et `PrimaryPropertyType` (3 « Low-Rise Multifamily » sont
classés à tort parmi les non résidentiels).

On travaille sur `df_clean`, une copie : `df_raw` reste intact pour pouvoir
comparer avant et après.

In [4]:
resid = (
    df_raw["BuildingType"].str.contains("Multifamily")
    | df_raw["PrimaryPropertyType"].str.contains("Multifamily")
)

df_clean = df_raw.loc[~resid].copy()
afficher_taille(df_clean, "Non résidentiels")

Non résidentiels : 1665 lignes, 46 colonnes


## 1.3. Colonnes inutiles

*Exploration, partie 1.2* : les colonnes qui n'ont qu'une seule valeur (ou
aucune) ne sont pas informatives. On les recalcule ici pour que le notebook
fonctionne tout seul.

In [5]:
cols_inutiles = df_raw.columns[df_raw.nunique() <= 1].tolist()
print(cols_inutiles)

df_clean = df_clean.drop(columns=cols_inutiles)
afficher_taille(df_clean, "Sans colonnes inutiles")

['DataYear', 'City', 'State', 'Comments']
Sans colonnes inutiles : 1665 lignes, 42 colonnes


# 2. Valeurs manquantes

## 2.1. Bâtiments sans relevé d'énergie

*Exploration, partie 2.3* : 2 bâtiments « Non-Compliant » n'ont ni
consommation ni émissions. On ne remplit jamais une cible, donc on supprime
ces lignes.

In [6]:
cibles = ["SiteEnergyUse(kBtu)", "TotalGHGEmissions"]

df_clean = df_clean.dropna(subset=cibles)
afficher_taille(df_clean, "Sans cible manquante")

Sans cible manquante : 1663 lignes, 42 colonnes


## 2.2. ZipCode et YearsENERGYSTARCertified

*Exploration, parties 2.2 et 2.7* :

- `ZipCode` est redondant avec la latitude et la longitude, qui sont plus
  précises
- `YearsENERGYSTARCertified` est presque vide, mal formatée, et dépend du
  score ENERGY STAR

In [7]:
df_clean = df_clean.drop(columns=["ZipCode", "YearsENERGYSTARCertified"])
afficher_taille(df_clean, "Sans ZipCode ni YearsENERGYSTARCertified")

Sans ZipCode ni YearsENERGYSTARCertified : 1663 lignes, 40 colonnes


## 2.3. Les 2e et 3e usages

*Exploration, partie 2.4* : un vide veut dire « pas de 2e ou 3e usage ». On
remplit avec « Aucun » pour le type et 0 pour la surface. Ce n'est pas une
imputation statistique : on traduit juste le sens du vide.

In [8]:
cols_usages_2_3 = [
    "SecondLargestPropertyUseType", "SecondLargestPropertyUseTypeGFA",
    "ThirdLargestPropertyUseType", "ThirdLargestPropertyUseTypeGFA",
]

df_clean = df_clean.fillna({
    "SecondLargestPropertyUseType": "Aucun",
    "SecondLargestPropertyUseTypeGFA": 0,
    "ThirdLargestPropertyUseType": "Aucun",
    "ThirdLargestPropertyUseTypeGFA": 0,
})
df_clean[cols_usages_2_3].isna().sum()

SecondLargestPropertyUseType       0
SecondLargestPropertyUseTypeGFA    0
ThirdLargestPropertyUseType        0
ThirdLargestPropertyUseTypeGFA     0
dtype: int64

## 2.4. L'usage principal

*Exploration, partie 2.5* : quand l'usage principal est vide et que la liste
ne contient qu'un seul usage, on le recopie depuis la liste. Pour la surface,
on prend celle du bâtiment (c'est une approximation).

Le dernier bâtiment (une dizaine d'usages) reste vide : on décidera au
moment de choisir les variables du modèle.

In [9]:
a_deduire = (
    df_clean["LargestPropertyUseType"].isna()
    & ~df_clean["ListOfAllPropertyUseTypes"].str.contains(",", na=False)
)
print(f"{a_deduire.sum()} usages principaux déduits de la liste")

df_clean.loc[a_deduire, "LargestPropertyUseType"] = (
    df_clean.loc[a_deduire, "ListOfAllPropertyUseTypes"]
)
df_clean.loc[a_deduire, "LargestPropertyUseTypeGFA"] = (
    df_clean.loc[a_deduire, "PropertyGFABuilding(s)"]
)

3 usages principaux déduits de la liste


## 2.5. ENERGY STAR Score

*Exploration, partie 2.6* : **on ne remplit pas**. Le score n'existe que
pour certains types de bâtiments : le remplir reviendrait à inventer des
valeurs. Il sera étudié à part dans la modélisation (comparaison avec et
sans, sur les bâtiments qui en ont un).

# 3. Erreurs et incohérences

## 3.1. Les quartiers

*Exploration, partie 3.1* : certains quartiers sont écrits de plusieurs
façons. On met tout en majuscules et on corrige Delridge.

In [10]:
df_clean["Neighborhood"] = (
    df_clean["Neighborhood"].str.upper()
    .replace("DELRIDGE NEIGHBORHOODS", "DELRIDGE")
)
df_clean["Neighborhood"].value_counts().sort_index()

Neighborhood
BALLARD                   70
CENTRAL                   56
DELRIDGE                  47
DOWNTOWN                 359
EAST                     120
GREATER DUWAMISH         346
LAKE UNION               147
MAGNOLIA / QUEEN ANNE    151
NORTH                     67
NORTHEAST                127
NORTHWEST                 86
SOUTHEAST                 46
SOUTHWEST                 41
Name: count, dtype: int64

## 3.2. Le nombre d'étages

*Exploration, partie 3.2* : 0 étage et 99 étages (l'église) sont des erreurs
de saisie. On les remplace par une valeur manquante. Elles seront remplies
avec la médiane dans le pipeline, après la séparation entre entraînement et
test.

`.mask(condition)` remplace la valeur par `NaN` là où la condition est
vraie, et garde la valeur ailleurs.

In [11]:
etages_faux = df_clean["NumberofFloors"].isin([0, 99])
print(f"{etages_faux.sum()} valeurs remplacées par NaN")

df_clean["NumberofFloors"] = df_clean["NumberofFloors"].mask(etages_faux)

17 valeurs remplacées par NaN


## 3.3. Le nombre de bâtiments

*Exploration, partie 3.2* : une propriété avec une surface et des étages a
au moins un bâtiment. On remplace 0 par 1, le minimum logique.

In [12]:
zero_batiment = df_clean["NumberofBuildings"] == 0
print(f"{zero_batiment.sum()} valeurs remplacées par 1")

df_clean.loc[zero_batiment, "NumberofBuildings"] = 1

52 valeurs remplacées par 1


## 3.4. Les cibles nulles ou négatives

*Exploration, partie 3.3* : les écoles « Missing Data » (déclaration
incomplète), le Whole Foods Interbay (émissions à 0) et le Bullitt Center
(émissions négatives) sont supprimés. Le modèle doit apprendre sur des
cibles fiables et représentatives.

In [13]:
cibles_ok = (df_clean[cibles] > 0).all(axis=1)
print(f"{(~cibles_ok).sum()} bâtiments supprimés")

df_clean = df_clean.loc[cibles_ok]
afficher_taille(df_clean, "Cibles strictement positives")

18 bâtiments supprimés
Cibles strictement positives : 1645 lignes, 40 colonnes


## 3.5. Les surfaces par usage

*Exploration, partie 3.4* : **pas de correction**. On ne sait pas quelle
surface est la bonne. Le choix de ces colonnes se fera au moment de la
sélection des variables.

# 4. Fuite de données

*Exploration, partie 4* : on supprime toutes les colonnes issues des relevés
d'énergie, car on ne les aurait pas pour un bâtiment jamais mesuré.

On garde :

- les 2 cibles : `SiteEnergyUse(kBtu)` et `TotalGHGEmissions`
- `ENERGYSTARScore`, qu'on doit étudier

In [14]:
cols_fuite = [
    "SiteEUI(kBtu/sf)", "SiteEUIWN(kBtu/sf)",
    "SourceEUI(kBtu/sf)", "SourceEUIWN(kBtu/sf)",
    "SiteEnergyUseWN(kBtu)", "SteamUse(kBtu)",
    "Electricity(kWh)", "Electricity(kBtu)",
    "NaturalGas(therms)", "NaturalGas(kBtu)",
    "GHGEmissionsIntensity",
]

df_clean = df_clean.drop(columns=cols_fuite)
afficher_taille(df_clean, "Sans colonnes de relevés")

Sans colonnes de relevés : 1645 lignes, 29 colonnes


# 5. Vérifications

On vérifie que chaque décision a bien été appliquée. `assert` arrête le
notebook avec une erreur si la condition est fausse : si tout passe, seul le
message final s'affiche.

In [15]:
cols_supprimees = (
    cols_inutiles + ["ZipCode", "YearsENERGYSTARCertified"] + cols_fuite
)

# Périmètre
assert not df_clean["BuildingType"].str.contains("Multifamily").any()
assert not df_clean["PrimaryPropertyType"].str.contains("Multifamily").any()

# Colonnes supprimées
assert not set(cols_supprimees) & set(df_clean.columns)

# Cibles remplies et strictement positives
assert (df_clean[cibles] > 0).all().all()

# 2e et 3e usages remplis
assert df_clean[cols_usages_2_3].notna().all().all()

# Erreurs corrigées
assert df_clean["Neighborhood"].nunique() == 13
assert not df_clean["NumberofFloors"].isin([0, 99]).any()
assert not (df_clean["NumberofBuildings"] == 0).any()

print("Toutes les vérifications sont passées")

Toutes les vérifications sont passées


**Les valeurs manquantes qui restent**

Elles doivent toutes correspondre à un choix fait pendant l'exploration.

In [16]:
restants = df_clean.isna().sum()
restants[restants > 0]

NumberofFloors                 17
LargestPropertyUseType          1
LargestPropertyUseTypeGFA       1
ENERGYSTARScore               555
Outlier                      1629
dtype: int64

| Colonne | Pourquoi c'est normal |
|---|---|
| `ENERGYSTARScore` | N'existe que pour certains types, étudié à part |
| `Outlier` | Vide = bâtiment non signalé comme anormal |
| `NumberofFloors` | Les 0 et 99 remplacés, médiane dans le pipeline |
| `LargestPropertyUseType` et sa surface | 1 bâtiment, décision lors du choix des variables |

**Bilan avant / après**

In [17]:
print(f"Lignes : {len(df_raw)} -> {len(df_clean)}")
print(f"Colonnes : {df_raw.shape[1]} -> {df_clean.shape[1]}")

Lignes : 3376 -> 1645
Colonnes : 46 -> 29


# 6. Export

On enregistre le fichier propre dans `data/processed/`. Le notebook de
modélisation repartira de ce fichier.

In [18]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

fichier_sortie = (
    DATA_PROCESSED / "2016-building-energy-benchmarking-clean.csv"
)
df_clean.to_csv(fichier_sortie, index=False)
print(f"Fichier exporté : {fichier_sortie.name}")

Fichier exporté : 2016-building-energy-benchmarking-clean.csv
